# Curated pan-zonal HepatoNet FastCORE workflow

Select **Runtime > Run all**. The repository files are loaded automatically from GitHub.


## 1 Setup

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/MatheusAmorim7/archivesforfastcore.git"
REPO_NAME = REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
REPO_DIR = Path("/content") / REPO_NAME

if "google.colab" in sys.modules:
    if REPO_DIR.exists():
        subprocess.run(
            ["git", "-C", str(REPO_DIR), "pull", "--ff-only"],
            check=True,
        )
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    print("Running from:", Path.cwd())
else:
    print("Local execution; current folder:", Path.cwd())


## 2 Dependencies

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "cobra>=0.30,<0.33",
        "python-libsbml>=5.20,<6",
        "swiglpk>=5.0,<6",
        "pandas>=2.0,<3",
        "numpy>=1.26,<3",
    ],
    check=True,
)
print("Dependencies installed.")


## 3 Files

In [ ]:
from pathlib import Path
import os
import sys

cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    ROOT = cwd.parent
elif (cwd / "config").exists() and (cwd / "notebooks").exists():
    ROOT = cwd
elif Path("/content/archivesforfastcore").exists():
    ROOT = Path("/content/archivesforfastcore")
else:
    raise FileNotFoundError(
        "Repository root not found. Run the GitHub setup cell first."
    )

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "notebooks"))

required = [
    ROOT / "config" / "hepatonet_original_13_08.xml",
    ROOT / "notebooks" / "run_pan_zonal_curated_fastcore.py",
    ROOT / "notebooks" / "fastcore_utils.py",
    ROOT / "notebooks" / "fastcore_outputs" / "reviewer_minimal_carbohydrate_core.csv",
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))

print("Project root:", ROOT)
print("All required files were found.")


## 4 Build

In [ ]:
import subprocess
import sys

command = [
    sys.executable,
    str(ROOT / "notebooks" / "run_pan_zonal_curated_fastcore.py"),
]
result = subprocess.run(
    command,
    cwd=ROOT,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
result.check_returncode()


## 5 Validation

In [ ]:
import pandas as pd

OUT = (
    ROOT
    / "notebooks"
    / "fastcore_outputs"
    / "pan_zonal_reviewer_core"
    / "curated_fastcore"
)
SUMMARY = OUT / "curated_fastcore_summary.csv"
VALIDATION = OUT / "curated_reduced_validation.csv"
BOUNDARIES = OUT / "curated_boundary_audit.csv"
FINAL_SBML = ROOT / "config" / "hepatonet_pan_zonal_curated_fastcore_reduced.xml"

summary = pd.read_csv(SUMMARY)
display(summary.T.rename(columns={0: "value"}))

row = summary.iloc[0]
assert int(row["prepared_reactions"]) == 2865
assert int(row["prepared_metabolites"]) == 1420
assert int(row["restored_irreversible_reactions"]) == 1106
assert int(row["final_reactions"]) == 91
assert int(row["final_metabolites"]) == 108
assert int(row["final_boundary_reactions"]) == 8
assert bool(row["functional_validation_all_optimal"])
assert row["runtime_objective_status"] == "optimal"
assert FINAL_SBML.exists()
print("Summary checks passed.")


## 6 Functional tests

In [ ]:
validation = pd.read_csv(VALIDATION)
display(validation)
assert (validation["status"] == "optimal").all()
assert set(validation["test"]) == {
    "glucose_uptake_to_g6p",
    "g6p_to_glucose_release",
    "lactate_to_pyruvate",
    "pyruvate_to_lactate",
    "oxidative_glucose_module",
}
print("All five functional tests are optimal.")


## 7 Boundary reactions

In [ ]:
boundaries = pd.read_csv(BOUNDARIES)
display(boundaries)
assert len(boundaries) == 8
assert not (boundaries["category"] == "unexpected").any()
print("Boundary audit passed.")


## 8 Final SBML

In [ ]:
import cobra

model = cobra.io.read_sbml_model(str(FINAL_SBML))
model.solver = "glpk"

required_reactions = {
    "r1032",
    "EX_HC00017_s",
    "EX_HC00040_s",
    "EX_HC00177_s",
    "EX_HC00021_s",
}
missing = required_reactions - {reaction.id for reaction in model.reactions}
assert not missing, f"Missing coupling reactions: {sorted(missing)}"
assert len(model.reactions) == 91
assert len(model.metabolites) == 108
assert model.objective.direction == "max"
assert model.reactions.get_by_id("r1032").bounds == (0.0, 1.0)

solution = model.optimize()
assert solution.status == "optimal"

print("Reactions:", len(model.reactions))
print("Metabolites:", len(model.metabolites))
print("Objective direction:", model.objective.direction)
print("r1032 bounds:", model.reactions.get_by_id("r1032").bounds)
print("Unconstrained runtime-objective test:", solution.status, solution.objective_value)


## 9 Download

In [ ]:
import hashlib
import shutil

def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

print("Final SBML:", FINAL_SBML)
print("SHA256:", sha256(FINAL_SBML))

archive_base = ROOT / "curated_fastcore_generated_outputs"
archive = shutil.make_archive(str(archive_base), "zip", root_dir=OUT)
print("Output archive:", archive)

if "google.colab" in sys.modules:
    from google.colab import files
    files.download(str(archive))
